In [42]:
import json
import os

base_path = "/kaggle/input/datasets/premshaw23/mmfundus-text"
file_path = os.path.join(base_path, "BRSET_16266.json")

with open(file_path, "r", encoding="utf-8") as f:
    brset = json.load(f)


def clinical_note_agent(record):
    """
    RetinaAgent Clinical-Note Agent.

    Returns a Grader-ready structured Python dictionary.
    Missing information is represented as None or [].
    No unsupported clinical information is inferred or fabricated.
    """

    diabetes_duration = None

    if record.get("diabetes_time_y") not in [None, "", "nan"]:
        try:
            diabetes_duration = float(record["diabetes_time_y"])
        except (ValueError, TypeError):
            diabetes_duration = None

    return {
        "image_id": record["ImageID"],

        "visual_acuity": {
            "right_eye": None,
            "left_eye": None
        },

        "lens_status": None,
        "prior_laser": None,
        "prior_anti_vegf": None,
        "hba1c": None,
        "diabetes_duration": diabetes_duration,
        "prior_vitrectomy": None,
        "symptoms": []
    }


In [43]:
for i in range(2):
    print("\n" + "=" * 70)
    print(f"RECORD {i}")

    note_output = clinical_note_agent(brset[i])

    print(json.dumps(note_output, indent=2))



RECORD 0
{
  "image_id": "/All_data/BRSET_16266/1.0.0/fundus_photos/img00001.jpg",
  "visual_acuity": {
    "right_eye": null,
    "left_eye": null
  },
  "lens_status": null,
  "prior_laser": null,
  "prior_anti_vegf": null,
  "hba1c": null,
  "diabetes_duration": 12.0,
  "prior_vitrectomy": null,
  "symptoms": []
}

RECORD 1
{
  "image_id": "/All_data/BRSET_16266/1.0.0/fundus_photos/img00002.jpg",
  "visual_acuity": {
    "right_eye": null,
    "left_eye": null
  },
  "lens_status": null,
  "prior_laser": null,
  "prior_anti_vegf": null,
  "hba1c": null,
  "diabetes_duration": 12.0,
  "prior_vitrectomy": null,
  "symptoms": []
}


In [44]:
import pandas as pd

SUBSET_SIZE = 500
subset = brset[:SUBSET_SIZE]

# Each element is the direct output of the Clinical-Note Agent.
note_agent_outputs = [
    clinical_note_agent(record)
    for record in subset
]

print("Number of Note Agent outputs:", len(note_agent_outputs))
print("\nFirst Note Agent output:")
display(note_agent_outputs[0])


Number of Note Agent outputs: 500

First Note Agent output:


{'image_id': '/All_data/BRSET_16266/1.0.0/fundus_photos/img00001.jpg',
 'visual_acuity': {'right_eye': None, 'left_eye': None},
 'lens_status': None,
 'prior_laser': None,
 'prior_anti_vegf': None,
 'hba1c': None,
 'diabetes_duration': 12.0,
 'prior_vitrectomy': None,
 'symptoms': []}

In [45]:
required_keys = [
    "image_id",
    "visual_acuity",
    "lens_status",
    "prior_laser",
    "prior_anti_vegf",
    "hba1c",
    "diabetes_duration",
    "prior_vitrectomy",
    "symptoms"
]

print("Schema check:")

for key in required_keys:
    print(f"{key:25s}:", key in note_agent_outputs[0])

print("\nVisual acuity structure:")
print(note_agent_outputs[0]["visual_acuity"])

print("\nMissing values remain explicit:")
print(note_agent_outputs[0])


Schema check:
image_id                 : True
visual_acuity            : True
lens_status              : True
prior_laser              : True
prior_anti_vegf          : True
hba1c                    : True
diabetes_duration        : True
prior_vitrectomy         : True
symptoms                 : True

Visual acuity structure:
{'right_eye': None, 'left_eye': None}

Missing values remain explicit:
{'image_id': '/All_data/BRSET_16266/1.0.0/fundus_photos/img00001.jpg', 'visual_acuity': {'right_eye': None, 'left_eye': None}, 'lens_status': None, 'prior_laser': None, 'prior_anti_vegf': None, 'hba1c': None, 'diabetes_duration': 12.0, 'prior_vitrectomy': None, 'symptoms': []}


In [46]:
# Direct interface for the next Grader Agent stage

note_output = clinical_note_agent(brset[0])

print("Note Agent output type:", type(note_output).__name__)
print("image_id:", note_output["image_id"])
print("Ready to pass directly into grader_agent(...).")


Note Agent output type: dict
image_id: /All_data/BRSET_16266/1.0.0/fundus_photos/img00001.jpg
Ready to pass directly into grader_agent(...).


In [47]:
# Example of the intended next-stage connection

# grader_output = grader_agent(
#     image_output=image_output,
#     note_output=note_output
# )

print("Next stage: Grader Agent")


Next stage: Grader Agent


In [48]:
# Final Note Agent output schema

print(note_agent_outputs[0])


{'image_id': '/All_data/BRSET_16266/1.0.0/fundus_photos/img00001.jpg', 'visual_acuity': {'right_eye': None, 'left_eye': None}, 'lens_status': None, 'prior_laser': None, 'prior_anti_vegf': None, 'hba1c': None, 'diabetes_duration': 12.0, 'prior_vitrectomy': None, 'symptoms': []}


In [49]:
required_fields = [
    "image_id",
    "visual_acuity",
    "lens_status",
    "prior_laser",
    "prior_anti_vegf",
    "hba1c",
    "diabetes_duration_years",
    "prior_vitrectomy",
    "symptoms",
    "image_quality",
    "missing_fields"
]

missing_columns = [
    col for col in required_fields
    if col not in note_agent_df.columns
]

print("Missing schema columns:", missing_columns)

Missing schema columns: []


In [50]:
print(
    "Records with missing fields:",
    note_agent_df["missing_fields"].apply(len).gt(0).sum()
)

print("\nExample missing fields:")
print(note_agent_df["missing_fields"].head(3).tolist())

Records with missing fields: 500

Example missing fields:
[['visual_acuity', 'lens_status', 'prior_laser', 'prior_anti_vegf', 'hba1c', 'prior_vitrectomy', 'symptoms'], ['visual_acuity', 'lens_status', 'prior_laser', 'prior_anti_vegf', 'hba1c', 'prior_vitrectomy', 'symptoms'], ['visual_acuity', 'lens_status', 'prior_laser', 'prior_anti_vegf', 'hba1c', 'prior_vitrectomy', 'symptoms']]


In [51]:
output_path = "/kaggle/working/note_agent_output.csv"

note_agent_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: /kaggle/working/note_agent_output.csv


In [52]:
display(note_agent_df.head(1).T)

,0
image_id,/All_data/BRSET_16266/1.0.0/fundus_photos/img0...
visual_acuity,None
lens_status,None
prior_laser,None
prior_anti_vegf,None
hba1c,None
diabetes_duration_years,12.0
prior_vitrectomy,None
symptoms,None
image_quality,Adequate
